# 🛒 Multimodal Search — Product Catalog

This notebook demonstrates **multi-embedding (multimodal) search** using pgVectorDB's `Spaces` system.

The idea: instead of encoding only text into vectors, we simultaneously encode **price**, **rating**, **category**, and **text** into separate embedding columns. At query time, we weight these signals dynamically.

### What You'll Learn
- `TextSpace` — semantic text embeddings
- `NumberSpace` — encode numeric values (price → prefer lower, rating → prefer higher)
- `CategorySpace` — one-hot encoding for filter-like behavior
- `multimodal_search()` — weighted multi-space search
- `rerank_search()` — retrieve-then-rerank with cross-encoders

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from langchain_core.documents import Document
from pgvectordb import (
    pgVectorDB, Config,
    TextSpace, NumberSpace, CategorySpace,
    NumberMode, DistanceMetric,
)

In [ ]:
embeddings = Config.get_embeddings()

rag = pgVectorDB(
    collection_name="nb_products",
    embedding_model=embeddings,
    connection_string=Config.get_connection_string(),
)
await rag.initialize(overwrite_existing=True)
print("✅ Initialized")

## 1. Define Vector Spaces

In [ ]:
spaces = [
    # Semantic embedding of product descriptions
    TextSpace(name="description", field="content"),
    
    # Price: lower is better (mode=MINIMUM)
    NumberSpace(
        name="price", field="price",
        min_value=0, max_value=1000,
        mode=NumberMode.MINIMUM,
    ),
    
    # Rating: higher is better (mode=MAXIMUM)
    NumberSpace(
        name="rating", field="rating",
        min_value=0, max_value=5,
        mode=NumberMode.MAXIMUM,
    ),
    
    # Category: one-hot encoding
    CategorySpace(
        name="category", field="category",
        categories=["electronics", "fashion", "home"],
    ),
]

rag.register_spaces(spaces)
print(f"✅ Registered {len(spaces)} spaces")
for s in spaces:
    print(f"   {s.name}: {s.dimensions} dims")

## 2. Add Products with Multi-Embeddings

In [ ]:
products = [
    Document(page_content="Sony WH-1000XM5 Wireless Noise Cancelling Headphones",
             metadata={"price": 349.99, "rating": 4.8, "category": "electronics", "brand": "Sony"}),
    Document(page_content="Bose QuietComfort 45 Bluetooth Wireless Headphones",
             metadata={"price": 279.00, "rating": 4.7, "category": "electronics", "brand": "Bose"}),
    Document(page_content="Apple AirPods Pro 2nd Gen Active Noise Cancellation",
             metadata={"price": 229.00, "rating": 4.6, "category": "electronics", "brand": "Apple"}),
    Document(page_content="Anker Soundcore Q45 Noise Cancelling Headphones",
             metadata={"price": 59.99, "rating": 4.3, "category": "electronics", "brand": "Anker"}),
    Document(page_content="KitchenAid Artisan Stand Mixer 5 Qt Stainless Steel",
             metadata={"price": 449.99, "rating": 4.9, "category": "home", "brand": "KitchenAid"}),
    Document(page_content="Instant Pot Duo Plus 9-in-1 Electric Pressure Cooker",
             metadata={"price": 99.99, "rating": 4.7, "category": "home", "brand": "Instant Pot"}),
    Document(page_content="Dyson V15 Detect Absolute Cordless Vacuum with Laser",
             metadata={"price": 749.99, "rating": 4.8, "category": "home", "brand": "Dyson"}),
    Document(page_content="Nike Air Max 270 Running Shoes Max Air Cushioning",
             metadata={"price": 150.00, "rating": 4.4, "category": "fashion", "brand": "Nike"}),
    Document(page_content="Levi's 511 Slim Fit Jeans Classic 5-Pocket Styling",
             metadata={"price": 49.99, "rating": 4.5, "category": "fashion", "brand": "Levi's"}),
]

ids = await rag.add_documents_multimodal(products, show_progress=True)
print(f"\n✅ Indexed {len(ids)} products with {len(spaces)} embedding columns each")

## 3. Build Multimodal Indexes

In [ ]:
indexes = await rag.build_multimodal_index(
    metric=DistanceMetric.COSINE,
    m=16, ef_construction=64,
)
print("✅ Indexes built:")
for name, info in indexes.items():
    print(f"   {name}: {info}")

## 4. Multimodal Search: Budget Headphones

Query: "noise cancelling headphones" — weighting description (50%), price (30%), rating (15%), category (5%).

In [ ]:
results = await rag.multimodal_search(
    query_params={
        "description": "noise cancelling wireless headphones",
        "price": 100.0,        # prefer ~$100
        "rating": 4.5,         # prefer high ratings
        "category": "electronics",
    },
    weights={
        "description": 0.50,
        "price": 0.30,
        "rating": 0.15,
        "category": 0.05,
    },
    k=5,
)

print("🎧 Budget Headphones (price-weighted):")
for i, r in enumerate(results, 1):
    m = r['metadata']
    print(f"  {i}. [{r['score']:.3f}] ${m.get('price','?')} ⭐{m.get('rating','?')} — {r['content'][:60]}")

## 5. Multimodal Search: Premium Home Appliance

Same system, different weights — prioritize rating over price.

In [ ]:
results = await rag.multimodal_search(
    query_params={
        "description": "premium kitchen appliance",
        "price": 500.0,
        "rating": 4.9,
        "category": "home",
    },
    weights={"description": 0.4, "price": 0.1, "rating": 0.4, "category": 0.1},
    k=3,
)

print("🏠 Premium Home Appliance (rating-weighted):")
for i, r in enumerate(results, 1):
    m = r['metadata']
    print(f"  {i}. [{r['score']:.3f}] ${m.get('price','?')} ⭐{m.get('rating','?')} — {r['content'][:60]}")

## 6. Pure Semantic vs Multimodal Comparison

Same query, but with **only** description weight = 1.0 — effectively pure semantic search.

In [ ]:
results = await rag.multimodal_search(
    query_params={"description": "noise cancelling wireless headphones"},
    weights={"description": 1.0},
    k=5,
)

print("🔊 Pure Semantic (description only):")
for i, r in enumerate(results, 1):
    m = r['metadata']
    print(f"  {i}. [{r['score']:.3f}] ${m.get('price','?')} — {r['content'][:60]}")
print("\n💡 Notice: without price weighting, the expensive Dyson ($750) and cheap Anker ($60) rank equally!")

## 7. Retrieve-Then-Rerank with Cross-Encoder

In [ ]:
try:
    from pgvectordb.rerankers import CrossEncoderReranker
    
    reranker = CrossEncoderReranker(model="cross-encoder/ms-marco-MiniLM-L-6-v2")
    
    reranked = await rag.rerank_search(
        query="affordable noise cancelling headphones with good value",
        reranker=reranker,
        k=9,             # Stage 1: fetch 9 candidates
        rerank_top_k=3,  # Stage 2: return best 3
        search_method="multimodal",
        query_params={
            "description": "affordable noise cancelling headphones good value",
            "price": 100.0,
            "category": "electronics",
        },
        weights={"description": 0.6, "price": 0.3, "category": 0.1},
    )
    
    print("🏆 Reranked Results:")
    for i, r in enumerate(reranked, 1):
        m = r['metadata']
        print(f"  {i}. [rerank={r['score']:.3f}] ${m.get('price','?')} — {r['content'][:60]}")
except ImportError:
    print("⚠ sentence-transformers not installed. Run: pip install sentence-transformers")

In [ ]:
await rag.delete_table()
await rag.close()
print("🧹 Cleaned up")